<a href="https://colab.research.google.com/github/pedromilken/Disciplinas-Doutorado/blob/main/aula0_2_support_vector_machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PGC305 - Tópicos especiais em LLM e Deep Learning

Neste notebook, você experimentará com um tipo de Support Vector Machine Linear.

## Preparação dos dados

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn import datasets

# Preparar o dataset
iris = datasets.load_iris()
X = iris.data; y = iris.target

X = X[y != 1] ; y = y[y != 1] # versicolor
y = torch.tensor(y, dtype=torch.float32)
y[y == 0] = -1  # SVM espera rótulos em {-1, +1}

X = torch.tensor(X, dtype=torch.float32) # Tensor é um tipo especial que suporta muitas dimensões

A nossa Support Vector Machine é basicamente um hiperplano definido por *w* e *b* que melhor separa as classes.

In [ ]:
# Definir parâmetros treináveis da Support Vector Machine: w e b
n_features = X.shape[1]
w = torch.randn(n_features, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# === Hiperparâmetros ===
learning_rate = 0.01
epochs = 300
optimizer = optim.Adam([w, b], lr=learning_rate)

## Execução do treinamento

In [ ]:
for epoch in range(epochs):
    optimizer.zero_grad()

    y_pred = X @ w + b  # Modelo SVM (um hiperplano que depende de w e b)

    # Hinge loss: max(0, 1 - y_i * (w^T x_i + b))
    perda_de_classificacao = torch.clamp(1 - y.view(-1, 1) * y_pred, min=0).mean()

    # Termo de regularização
    perda_de_distancia_entre_classes = 0.5 * torch.sum(w ** 2) # 2/||w|| é a distância que queremos que seja a maior possível

    # Função objetivo tradicional: minimizar reg + C * hinge
    loss = perda_de_distancia_entre_classes + perda_de_classificacao

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss={loss.item():.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn import datasets

# 1. Dados e Normalização
iris = datasets.load_iris()
X = torch.tensor(iris.data[iris.target != 1], dtype=torch.float32)
y = torch.tensor(iris.target[iris.target != 1], dtype=torch.float32)
y[y == 0] = -1
y[y == 2] = 1 # Garante +1 e -1
X = (X - X.mean(0)) / X.std(0)

# 2. Modelo
model = nn.Linear(X.shape[1], 1)
optimizer = optim.Adam(model.parameters(), lr=0.01)
C = 10.0 # Parâmetro de regularização

for epoch in range(500):
    optimizer.zero_grad()
    outputs = model(X).flatten()

    # Hinge Loss + L2 Regularization
    hinge_loss = torch.mean(torch.clamp(1 - y * outputs, min=0))
    l2_reg = 0.5 * torch.sum(model.weight ** 2)

    loss = l2_reg + C * hinge_loss
    loss.backward()
    optimizer.step()

print(f"Treinamento finalizado. Loss: {loss.item():.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_svc_decision_boundary(X, y, w, b):
    # Converter para numpy para o matplotlib
    X_np = X.detach().numpy()
    y_np = y.detach().numpy()
    w_np = w.detach().numpy().flatten()
    b_np = b.detach().numpy()

    plt.figure(figsize=(10, 6))

    # Plota os pontos: Setosa (-1) e Virginica (1)
    plt.scatter(X_np[y_np == -1, 0], X_np[y_np == -1, 1], color='red', label='Setosa (-1)')
    plt.scatter(X_np[y_np == 1, 0], X_np[y_np == 1, 1], color='blue', label='Virginica (1)')

    # Calcula a linha de decisão: w0*x + w1*y + b = 0  => y = -(w0/w1)*x - (b/w1)
    x_axis = np.linspace(X_np[:, 0].min(), X_np[:, 0].max(), 100)
    decision_boundary = -(w_np[0] / w_np[1]) * x_axis - (b_np / w_np[1])

    # Calcula as margens (distância 1 do hiperplano)
    margin_up = decision_boundary + 1 / w_np[1]
    margin_down = decision_boundary - 1 / w_np[1]

    plt.plot(x_axis, decision_boundary, 'k-', label='Hiperplano de Decisão')
    plt.plot(x_axis, margin_up, 'k--', alpha=0.5, label='Margem (+1 / -1)')
    plt.plot(x_axis, margin_down, 'k--', alpha=0.5)

    plt.ylim(X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5)
    plt.xlabel('Comprimento da Sépala (Normalizado)')
    plt.ylabel('Largura da Sépala (Normalizado)')
    plt.legend()
    plt.title('Visualização do SVM Linear')
    plt.show()

# Nota: Para este gráfico funcionar bem, treine o modelo usando apenas X[:, :2]
# (as duas primeiras colunas) para que w tenha apenas 2 pesos.

# 1. Extrair w e b do modelo treinado (nn.Linear)
w = model.weight.data.flatten()
b = model.bias.data

# 2. Chamar a função de plotagem
# Importante: X deve ter apenas 2 colunas para o gráfico fazer sentido!
plot_svc_decision_boundary(X[:, :2], y, w, b)